- Author: Yousef Al Zeer

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns',100)
import missingno
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
# Set pandas as the default output for sklearn
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
# load data

path='https://docs.google.com/spreadsheets/d/e/2PACX-1vSlxm22ftBEbkw8AS5NSzggxMWRLD_gxJ8o6RTZqdrTcQXcEt3EpsShuEBbSWDSmWgB_xLgSDn2fxDH/pub?output=csv'
df = pd.read_csv(path)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      77 non-null     object 
 1   mfr       75 non-null     object 
 2   type      77 non-null     object 
 3   calories  72 non-null     float64
 4   protein   77 non-null     int64  
 5   fat       70 non-null     float64
 6   sodium    77 non-null     int64  
 7   fiber     71 non-null     float64
 8   carbo     77 non-null     float64
 9   sugars    71 non-null     float64
 10  potass    77 non-null     int64  
 11  vitamins  77 non-null     int64  
 12  shelf     75 non-null     object 
 13  weight    77 non-null     float64
 14  cups      77 non-null     float64
 15  rating    77 non-null     float64
dtypes: float64(8), int64(4), object(4)
memory usage: 9.8+ KB


,name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating
0,100% Bran,N,C,NaN,4,1.0,130,10.0,5.0,6.0,280,25,top,1.0,0.33,68.402973
1,100% Natural Bran,Q,C,120.0,3,5.0,15,2.0,8.0,8.0,135,0,top,1.0,1.00,33.983679
2,All-Bran,K,C,70.0,4,1.0,260,9.0,7.0,5.0,320,25,top,1.0,0.33,59.425505
3,All-Bran with Extra Fiber,K,C,50.0,4,0.0,140,14.0,8.0,0.0,330,25,top,1.0,0.50,93.704912
4,Almond Delight,R,C,NaN,2,2.0,200,1.0,14.0,8.0,-1,25,NaN,1.0,0.75,34.384843


In [ ]:
df.nunique()/len(df) * 100

,0
name,100.000000
mfr,9.090909
type,2.597403
calories,14.285714
protein,7.792208
fat,6.493506
sodium,35.064935
fiber,16.883117
carbo,28.571429
sugars,22.077922


In [ ]:
target = 'rating'
features = ['mfr', 'type', 'calories', 'protein', 'fat', 'fiber', 'sugars', 'shelf']
X = df[features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 57 entries, 30 to 51
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   mfr       55 non-null     object 
 1   type      57 non-null     object 
 2   calories  57 non-null     float64
 3   protein   57 non-null     int64  
 4   fat       52 non-null     float64
 5   fiber     52 non-null     float64
 6   sugars    52 non-null     float64
 7   shelf     57 non-null     object 
dtypes: float64(4), int64(1), object(3)
memory usage: 4.0+ KB


In [ ]:
X_train['shelf'].value_counts(dropna=False)

,count
shelf,
top,26
bottom,17
middle,14


## Create Pipelines and Define Tuples

### Ordinal

In [ ]:
ord_cols = ['shelf']

impute_common = SimpleImputer(strategy='most_frequent')

shelf_order = ['bottom','middle','top']
ord_encoder = OrdinalEncoder(categories=[shelf_order])

scaler = StandardScaler()

# pipeline
ord_pipe = make_pipeline(impute_common, ord_encoder, scaler)
ord_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                ('ordinalencoder',
                 OrdinalEncoder(categories=[['bottom', 'middle', 'top']])),
                ('standardscaler', StandardScaler())])

In [ ]:
ord_tuple = ('ordinal' , ord_pipe , ord_cols)
ord_tuple

('ordinal',
 Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                 ('ordinalencoder',
                  OrdinalEncoder(categories=[['bottom', 'middle', 'top']])),
                 ('standardscaler', StandardScaler())]),
 ['shelf'])

### Numeric

In [ ]:
num_cols = X_train.select_dtypes('number').columns

impute_mean = SimpleImputer(strategy='mean')
scaler = StandardScaler()


#pipeline
num_pipe = make_pipeline(impute_mean , scaler)
num_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer()),
                ('standardscaler', StandardScaler())])

In [ ]:
num_tuple = ("Numeric" , num_pipe , num_cols)
num_tuple

('Numeric',
 Pipeline(steps=[('simpleimputer', SimpleImputer()),
                 ('standardscaler', StandardScaler())]),
 Index(['calories', 'protein', 'fat', 'fiber', 'sugars'], dtype='object'))

### Categorical

In [ ]:
cat_cols = X_train.select_dtypes('object').drop(columns=ord_cols).columns

impute_missing = SimpleImputer(strategy='constant',fill_value='MISSING')
ohe_encoder = OneHotEncoder(sparse_output=False , handle_unknown='ignore')

ohe_pipe = make_pipeline(impute_missing , ohe_encoder)
ohe_pipe

Pipeline(steps=[('simpleimputer',
                 SimpleImputer(fill_value='MISSING', strategy='constant')),
                ('onehotencoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [ ]:
cat_cols

Index(['mfr', 'type'], dtype='object')

In [ ]:
ohe_tuple = ("Categorical" , ohe_pipe , cat_cols)
ohe_tuple

('Categorical',
 Pipeline(steps=[('simpleimputer',
                  SimpleImputer(fill_value='MISSING', strategy='constant')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['mfr', 'type'], dtype='object'))

## Instantiate the ColumnTransformer

In [ ]:
col_transformer = ColumnTransformer([num_tuple , ord_tuple , ohe_tuple ], verbose_feature_names_out=False)

In [ ]:
col_transformer.fit(X_train)

ColumnTransformer(transformers=[('Numeric',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['calories', 'protein', 'fat', 'fiber', 'sugars'], dtype='object')),
                                ('ordinal',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(categories=[['bottom',
                                                                              'middle',
                                                                              'top']])),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 ['shelf']),
                                ('Categorical',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='MISSING',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['mfr', 'type'], dtype='object'))],
                  verbose_feature_names_out=False)

In [ ]:
X_train_processed = col_transformer.transform(X_train)
X_test_processed = col_transformer.transform(X_test)
print("training set processed : \n")
display(X_train_processed.head())
print("\n")
print("testing set processed : \n")
display(X_test_processed.head())



training set processed : 



,calories,protein,fat,fiber,sugars,shelf,mfr_A,mfr_G,mfr_K,mfr_MISSING,mfr_N,mfr_P,mfr_Q,mfr_R,type_C,type_H
30,-0.319703,-0.524507,-1.007451,-0.871334,1.992024,-1.355719,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
40,0.172812,-0.524507,0.040298,-0.871334,-0.795023,-0.184871,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
39,1.650358,0.354813,0.040298,-0.018050,0.598501,0.985978,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
16,-0.319703,-0.524507,-1.007451,-0.444692,-1.027277,-1.355719,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
65,-0.812218,0.354813,-1.007451,0.408592,-1.491785,-1.355719,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0




testing set processed : 



,calories,protein,fat,fiber,sugars,shelf,mfr_A,mfr_G,mfr_K,mfr_MISSING,mfr_N,mfr_P,mfr_Q,mfr_R,type_C,type_H
4,0.0,-0.524507,1.088047,-0.444692,0.366247,0.985978,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
35,0.0,-1.403826,1.088047,-0.444692,1.063009,0.985978,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
10,0.0,-1.403826,1.088047,-0.871334,1.295263,-0.184871,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
0,0.0,1.234133,0.040298,3.395084,-0.098261,0.985978,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
45,0.0,1.234133,2.135796,0.408592,1.063009,0.985978,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
